# Smart Finance - Análise de Comportamento Financeiro

# Engenharia de Atributos

## Objetivo

Criar atributos derivados que representem o comportamento financeiro dos usuários.


In [6]:
# ==========================================================
# IMPORTAÇÃO DAS BIBLIOTECAS
# ==========================================================

import pandas as pd
import numpy as np

In [7]:
# ==========================================================
# LEITURA DOS DATASETS
# ==========================================================

#usuarios = pd.read_csv(
#    "data/processed/usuarios_processado.csv"
#)

#transacoes = pd.read_csv(
#    "data/processed/transacoes_processado.csv"
#)

In [8]:
# ============================================================
# LEITURA DOS DATASETS (colab)
# ============================================================

usuarios = pd.read_csv(f"{BASE}/processed/usuarios_processado.csv")
transacoes = pd.read_csv(f"{BASE}/processed/transacoes_processado.csv")

In [9]:
# ============================================================
# LEITURA DOS DATASETS (DRIVE)
# ============================================================
from google.colab import drive
drive.mount('/content/drive')
BASE = "/content/drive/MyDrive/Hackatona G9 Finance AI/data"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
# ==========================================================
# SEPARAÇÃO ENTRE RECEITAS E DESPESAS
# ==========================================================

despesas = transacoes[
    transacoes["tipo_transacao"] == "Despesa"
]

receitas = transacoes[
    transacoes["tipo_transacao"] == "Receita"
]

In [11]:
# ==========================================================
# TOTAL DE DESPESAS
# ==========================================================

gasto_total = (

    despesas

    .groupby("usuario_id")["valor_transacao"]

    .sum()

    .rename("gasto_total")

)

In [12]:
# ==========================================================
# TOTAL DE RECEITAS
# ==========================================================

receita_total = (

    receitas

    .groupby("usuario_id")["valor_transacao"]

    .sum()

    .rename("receita_total")

)

In [13]:
# ==========================================================
# QUANTIDADE DE TRANSAÇÕES
# ==========================================================

numero_transacoes = (

    transacoes

    .groupby("usuario_id")

    .size()

    .rename("numero_transacoes")

)

In [14]:
# ==========================================================
# TICKET MÉDIO
# ==========================================================

ticket_medio = (

    despesas

    .groupby("usuario_id")["valor_transacao"]

    .mean()

    .rename("ticket_medio")

)

In [15]:
# ==========================================================
# MAIOR DESPESA
# ==========================================================

maior_despesa = (

    despesas

    .groupby("usuario_id")["valor_transacao"]

    .max()

    .rename("maior_despesa")

)

In [16]:
# ==========================================================
# MENOR DESPESA
# ==========================================================

menor_despesa = (

    despesas

    .groupby("usuario_id")["valor_transacao"]

    .min()

    .rename("menor_despesa")

)

In [17]:
# ==========================================================
# UNIÃO DAS FEATURES
# ==========================================================

features_financeiras = pd.concat(

    [

        gasto_total,

        receita_total,

        numero_transacoes,

        ticket_medio,

        maior_despesa,

        menor_despesa

    ],

    axis=1

)

In [18]:
# ==========================================================
# PREENCHIMENTO DE VALORES AUSENTES
# ==========================================================

features_financeiras = (

    features_financeiras

    .fillna(0)

)

In [19]:
# ==========================================================
# SALDO FINANCEIRO
# ==========================================================

features_financeiras["saldo"] = (

    features_financeiras["receita_total"]

    -

    features_financeiras["gasto_total"]

)

In [20]:
# ==========================================================
# RAZÃO GASTO / RENDA
# ==========================================================

features_financeiras = (

    features_financeiras

    .merge(

        usuarios[

            [

                "usuario_id",

                "renda_mensal"

            ]

        ],

        on="usuario_id"

    )

)

In [21]:
features_financeiras["razao_gasto_renda"] = (

    features_financeiras["gasto_total"]

    /

    features_financeiras["renda_mensal"]

)

In [22]:
# ==========================================================
# GASTOS POR CATEGORIA
# ==========================================================

gastos_categoria = (

    despesas

    .pivot_table(

        index="usuario_id",

        columns="categoria",

        values="valor_transacao",

        aggfunc="sum",

        fill_value=0

    )

)

In [23]:
# ==========================================================
# RENOMEAR COLUNAS
# ==========================================================

gastos_categoria.columns = [

    f"gasto_{col.lower()}"

    for col in gastos_categoria.columns

]

In [24]:
# ==========================================================
# UNIR COM AS FEATURES FINANCEIRAS
# ==========================================================

features_financeiras = (

    features_financeiras

    .merge(

        gastos_categoria,

        on="usuario_id",

        how="left"

    )

)

In [25]:
# ==========================================================
# PREENCHER VALORES NULOS
# ==========================================================

features_financeiras.fillna(0, inplace=True)

In [26]:
# ==========================================================
# PERCENTUAL GASTO POR CATEGORIA
# ==========================================================

colunas_gastos = [

    coluna

    for coluna in features_financeiras.columns

    if coluna.startswith("gasto_")

]

In [27]:
for coluna in colunas_gastos:

    percentual = coluna.replace(

        "gasto",

        "perc"

    )

    features_financeiras[percentual] = np.where(

        features_financeiras["gasto_total"] > 0,

        features_financeiras[coluna]

        /

        features_financeiras["gasto_total"],

        0

    )

In [28]:
# ==========================================================
# QUANTIDADE DE TRANSAÇÕES RECORRENTES
# ==========================================================

recorrentes = (

    transacoes

    .groupby("usuario_id")["transacao_recorrente"]

    .sum()

    .rename("qtd_recorrentes")

)

In [29]:
features_financeiras = (

    features_financeiras

    .merge(

        recorrentes,

        on="usuario_id",

        how="left"

    )

)

In [30]:
# ==========================================================
# QUANTIDADE DE COMPRAS PARCELADAS
# ==========================================================

parceladas = (

    transacoes

    .groupby("usuario_id")["parcelado"]

    .sum()

    .rename("qtd_parceladas")

)

In [31]:
features_financeiras = (

    features_financeiras

    .merge(

        parceladas,

        on="usuario_id",

        how="left"

    )

)

In [32]:
# ==========================================================
# PROPORÇÃO DAS FORMAS DE PAGAMENTO
# ==========================================================

formas_pagamento = pd.crosstab(

    transacoes["usuario_id"],
    transacoes["forma_pagamento"],
    normalize="index"

)

In [33]:
formas_pagamento.columns = [

    f"perc_{col.lower()}"

    for col in formas_pagamento.columns

]

formas_pagamento.reset_index(inplace=True)

In [34]:
features_financeiras = (

    features_financeiras

    .merge(

        formas_pagamento,

        on="usuario_id",

        how="left"

    )

    .fillna(0)

)

In [35]:
# ==========================================================
# DATASET DO MODELO 1
# Classificação das Transações
# ==========================================================

dataset_modelo_categoria = transacoes[

    [

        "descricao_transacao",

        "valor_transacao",

        "categoria"

    ]

].copy()

In [36]:
dataset_modelo_categoria.drop_duplicates(

    inplace=True

)

dataset_modelo_categoria.reset_index(

    drop=True,

    inplace=True

)

In [37]:
dataset_modelo_categoria.head()

,descricao_transacao,valor_transacao,categoria
0,Compra online - Burger King (compra à vista),829.09,Alimentação
1,Transação - Renner (parcelado em 4x),232.37,Compras
2,Consumo em CDB Banco Inter (compra à vista),910.31,Investimentos
3,Débito - Decolar.com,421.92,Lazer
4,Pagamento de conta - Açougue Central,450.77,Alimentação


In [38]:
# ==========================================================
# DATASET DO MODELO 2
# Perfil Financeiro
# ==========================================================

dataset_modelo_perfil = (

    usuarios.merge(

        features_financeiras,

        on="usuario_id",

        how="left"

    )

)

In [39]:
dataset_modelo_perfil.fillna(0, inplace=True)

In [40]:
dataset_modelo_perfil.head()

,usuario_id,idade,sexo,estado_civil,dependentes,cidade,estado,profissao,escolaridade,renda_mensal_x,...,perc_transporte,qtd_recorrentes,qtd_parceladas,perc_boleto,perc_cartão de crédito,perc_cartão de débito,perc_dinheiro,perc_débito automático,perc_pix,perc_transferência bancária
0,1,40,Masculino,Divorciado,2,Belo Horizonte,MG,Analista de TI,Médio,9214.53,...,0.050941,7,3,0.090909,0.295455,0.227273,0.022727,0.022727,0.204545,0.136364
1,2,41,Masculino,Solteiro,2,Duque de Caxias,RJ,Recepcionista,Fundamental,1812.22,...,0.001064,7,1,0.151515,0.393939,0.060606,0.030303,0.030303,0.212121,0.121212
2,3,40,Feminino,Casado,0,Guarulhos,SP,Analista de TI,Superior,7668.81,...,0.012108,12,0,0.074074,0.240741,0.166667,0.129630,0.074074,0.166667,0.148148
3,4,38,Masculino,Divorciado,4,Cuiabá,MT,Analista de TI,Superior,7828.20,...,0.016859,3,2,0.040000,0.280000,0.200000,0.000000,0.080000,0.160000,0.240000
4,5,27,Feminino,Solteiro,0,Maceió,AL,Empresário,Superior,21743.69,...,0.034179,14,0,0.040816,0.183673,0.142857,0.000000,0.102041,0.346939,0.183673


In [41]:
# ==========================================================
# VERIFICAÇÃO FINAL
# ==========================================================

print(dataset_modelo_categoria.shape)

print(dataset_modelo_perfil.shape)

(39935, 3)
(1000, 65)


In [42]:
print(dataset_modelo_categoria.info())

print()

print(dataset_modelo_perfil.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39935 entries, 0 to 39934
Data columns (total 3 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   descricao_transacao  39935 non-null  object 
 1   valor_transacao      39935 non-null  float64
 2   categoria            39935 non-null  object 
dtypes: float64(1), object(2)
memory usage: 936.1+ KB
None

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 65 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   usuario_id                   1000 non-null   int64  
 1   idade                        1000 non-null   int64  
 2   sexo                         1000 non-null   object 
 3   estado_civil                 1000 non-null   object 
 4   dependentes                  1000 non-null   int64  
 5   cidade                       1000 non-null   object 
 6   estado 

In [43]:
print(

dataset_modelo_categoria.isnull().sum().sum()

)

print(

dataset_modelo_perfil.isnull().sum().sum()

)

0
0


In [44]:
# ==========================================================
# EXPORTAÇÃO
# ==========================================================

import os

os.makedirs(

    "data/model",

    exist_ok=True

)

In [45]:
#dataset_modelo_categoria.to_csv(

#    "data/model/dataset_modelo_categoria.csv",

#    index=False,

#    encoding="utf-8"

#)

#dataset_modelo_perfil.to_csv(

#    "data/model/dataset_modelo_perfil.csv",

#    index=False,

#    encoding="utf-8"

#)

In [48]:
dataset_modelo_categoria.to_csv(
    f"{BASE}/model/dataset_modelo_categoria.csv",
    index=False,
    encoding="utf-8"
)

dataset_modelo_perfil.to_csv(
    f"{BASE}/model/dataset_modelo_perfil.csv",
    index=False,
    encoding="utf-8"
)

print("Datasets gerados com sucesso!")

Datasets gerados com sucesso!


# Conclusão

Nesta etapa foi realizada a Engenharia de Atributos do projeto.

Foram criadas variáveis financeiras e comportamentais derivadas das transações e dos dados cadastrais dos usuários.

Foram gerados dois datasets distintos:

- **dataset_modelo_categoria.csv**: utilizado para o treinamento do modelo de classificação automática de transações.

- **dataset_modelo_perfil.csv**: utilizado para o treinamento do modelo de classificação do perfil financeiro do usuário.

Esses arquivos serão utilizados no treinamento, avaliação e serialização dos modelos de Machine Learning.